In [0]:
%sql

SELECT * FROM bakery_pos_develop.bronze.pos_session limit 10

In [0]:
%sql

DESCRIBE bakery_pos_develop.bronze.pos_session

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stg_pos_session AS
SELECT
    id AS session_id,
    name AS session_name,
    config_id[0] AS config_id,
    config_id[1] AS config_name,
    start_at,
    stop_at,
    cash_register_balance_end_real,
    cash_register_balance_start,
    CAST(create_date AS TIMESTAMP) AS created_at,
    CAST(write_date AS TIMESTAMP) AS updated_at,
    processing_timestamp AS bronze_ingested_at
FROM bakery_pos_develop.bronze.pos_session
QUALIFY ROW_NUMBER() OVER(
    PARTITION BY id
    ORDER BY write_date DESC,
    processing_timestamp DESC
) = 1


In [0]:
%sql

CREATE TABLE IF NOT EXISTS bakery_pos_develop.silver.pos_session (
    session_id BIGINT,
    session_name STRING,
    config_id INT,
    config_name STRING,
    start_at TIMESTAMP,
    stop_at TIMESTAMP,
    cash_register_balance_end_real DOUBLE,
    cash_register_balance_start DOUBLE,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    bronze_ingested_at TIMESTAMP,
    silver_processed_at TIMESTAMP,
    is_updated BOOLEAN,
    is_deleted BOOLEAN
)

In [0]:
%sql

MERGE INTO bakery_pos_develop.silver.pos_session AS target
USING (
  SELECT *, current_timestamp() AS silver_processed_at 
  FROM stg_pos_session
) AS source
ON target.session_id = source.session_id
WHEN MATCHED AND source.updated_at > target.updated_at THEN
  UPDATE SET
    session_name = source.session_name,
    config_id = source.config_id,
    config_name = source.config_name,
    start_at = source.start_at,
    stop_at = source.stop_at,
    cash_register_balance_end_real = source.cash_register_balance_end_real,
    cash_register_balance_start = source.cash_register_balance_start,
    created_at = source.created_at,
    updated_at = source.updated_at,
    is_updated = True,
    is_deleted = False,
    bronze_ingested_at = source.bronze_ingested_at,
    silver_processed_at = source.silver_processed_at
WHEN NOT MATCHED THEN
  INSERT (
    session_id,
    session_name,
    config_id,
    config_name,
    start_at,
    stop_at,
    cash_register_balance_end_real,
    cash_register_balance_start,
    created_at,
    updated_at,
    bronze_ingested_at,
    silver_processed_at,
    is_updated,
    is_deleted
  )
  VALUES (
    source.session_id,
    source.session_name,
    source.config_id,
    source.config_name,
    source.start_at,
    source.stop_at,
    source.cash_register_balance_end_real,
    source.cash_register_balance_start,
    source.created_at,
    source.updated_at,
    source.bronze_ingested_at,
    source.silver_processed_at,
    False,
    False
  );